# Catalogs from Python

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

SciQLop exposes a Python CRUD API for the same catalogs you see in the catalog browser. Use it to programmatically build, share, or post-process collections of time intervals.

Catalogs created here appear in the catalog browser tree under their provider, can be overlaid on plot panels via drag-and-drop, and can be edited from the GUI (see the [GUI tour](1-SciQLopGUIDiscovery.ipynb)). Round-tripping is preserved: an event saved by Python and reloaded by Python keeps its UUID and metadata.

**What you'll learn**
- List catalogs, optionally filtered by provider prefix.
- Create, save (upsert), add events to, remove events from, and delete catalogs.
- Identify events by their UUID across calls.

**Prerequisites** — [Speasy catalogs](../Speasy/3-SpeasyCatalogs.ipynb).

**Next** — [Graphic primitives](10-SciQLopGraphicPrimitives.ipynb) (related: annotating events on plots).


In [ ]:
from datetime import datetime, timezone
from SciQLop.user_api.catalogs import catalogs

## 1. Listing available catalogs

Use `catalogs.list()` to see all catalogs, or pass a prefix to filter by provider.

In [ ]:
# List all catalogs from all providers
all_catalogs = catalogs.list()
print(f"Total catalogs: {len(all_catalogs)}")
for path in all_catalogs[:10]:
    print(f"  {path}")

In [ ]:
# Filter by provider
local_catalogs = catalogs.list("My Catalogs")
print(f"Local catalogs: {len(local_catalogs)}")

## 2. Creating a catalog

Use `catalogs.create()` to make a new catalog. Events can be:
- A list of `(start, stop)` tuples
- A list of `(start, stop, meta_dict)` tuples
- A `speasy.Catalog` object

In [ ]:
events = [
    (datetime(2015, 11, 18, 2, 20, 0, tzinfo=timezone.utc),
     datetime(2015, 11, 18, 2, 25, 0, tzinfo=timezone.utc),
     {"type": "magnetopause_crossing", "quality": "good"}),

    (datetime(2015, 11, 18, 2, 45, 0, tzinfo=timezone.utc),
     datetime(2015, 11, 18, 2, 50, 0, tzinfo=timezone.utc),
     {"type": "magnetopause_crossing", "quality": "fair"}),

    (datetime(2015, 11, 18, 3, 10, 0, tzinfo=timezone.utc),
     datetime(2015, 11, 18, 3, 15, 0, tzinfo=timezone.utc),
     {"type": "flux_transfer_event", "quality": "good"}),
]

catalogs.create("My Catalogs//my_events", events)
print("Catalog created!")

## 3. Retrieving a catalog

`catalogs.get()` returns a `speasy.Catalog` object. Each event carries a
`meta["__sciqlop_uuid__"]` for round-trip identification.

In [ ]:
cat = catalogs.get("My Catalogs//my_events")

print(f"Catalog: {cat.name}, {len(cat)} events")
for event in cat:
    print(f"  {event.start_time} -> {event.stop_time}  meta={dict(event.meta)}")

## 4. Adding events incrementally

Use `catalogs.add_events()` to append new events without replacing the existing ones.

In [ ]:
new_events = [
    (datetime(2015, 11, 18, 3, 20, 0, tzinfo=timezone.utc),
     datetime(2015, 11, 18, 3, 22, 0, tzinfo=timezone.utc),
     {"type": "jet", "quality": "good"}),
]

catalogs.add_events("My Catalogs//my_events", new_events)

cat = catalogs.get("My Catalogs//my_events")
print(f"Now {len(cat)} events")

## 5. Removing specific events

Events are identified by their UUID (from `meta["__sciqlop_uuid__"]`).
You can pass `speasy.Event` objects directly, or raw UUID strings.

In [ ]:
cat = catalogs.get("My Catalogs//my_events")

# Remove the last event
events_to_remove = [cat[-1]]
catalogs.remove_events("My Catalogs//my_events", events_to_remove)

cat = catalogs.get("My Catalogs//my_events")
print(f"After removal: {len(cat)} events")

## 6. Replacing all events (bulk save)

`catalogs.save()` replaces all events in a catalog. If the catalog doesn't exist, it's created (upsert).

In [ ]:
replacement_events = [
    (datetime(2015, 11, 18, 2, 30, 0, tzinfo=timezone.utc),
     datetime(2015, 11, 18, 2, 35, 0, tzinfo=timezone.utc)),
]

catalogs.save("My Catalogs//my_events", replacement_events)

cat = catalogs.get("My Catalogs//my_events")
print(f"After bulk save: {len(cat)} events")

## 7. Deleting a catalog

In [ ]:
catalogs.remove("My Catalogs//my_events")
print("Catalog deleted.")

## API reference

| Method | Purpose |
|--------|--------|
| `catalogs.list(prefix=None)` | List catalog paths, optionally filtered |
| `catalogs.get(path)` | Retrieve as `speasy.Catalog` |
| `catalogs.create(path, data)` | Create new catalog (error if exists) |
| `catalogs.save(path, data)` | Replace all events (upsert) |
| `catalogs.add_events(path, data)` | Append events to existing catalog |
| `catalogs.remove_events(path, events)` | Remove specific events by UUID |
| `catalogs.remove(path)` | Delete a catalog |

**Path format:** `"provider//sub//path//catalog_name"` (e.g. `"My Catalogs//my_catalog"`)

**Data formats:** `speasy.Catalog`, list of `(start, stop)`, or list of `(start, stop, meta_dict)`